In [ ]:
import os

os.chdir("..")
os.getcwd()

In [ ]:
import open_clip
import torch
from huggingface_hub import hf_hub_download
from PIL import Image

In [ ]:
for model_name in ["RN50", "ViT-B-32", "ViT-L-14"]:
    checkpoint_path = hf_hub_download(
        "chendelong/RemoteCLIP",
        f"RemoteCLIP-{model_name}.pt",
        cache_dir="/Users/gabriele/.cache/huggingface/hub/",
    )
    print(f"{model_name} is downloaded to {checkpoint_path}.")

In [ ]:
model_name = "ViT-L-14"  # 'RN50' or 'ViT-B-32' or 'ViT-L-14'
model, _, preprocess = open_clip.create_model_and_transforms(model_name)
tokenizer = open_clip.get_tokenizer(model_name)


ckpt = torch.load(
    f"{os.path.dirname(checkpoint_path)}/RemoteCLIP-{model_name}.pt", map_location="cpu"
)
message = model.load_state_dict(ckpt)
print(message)

model = model.to("mps").eval()

preprocess

In [ ]:
text_queries = [
    "A busy airport with many airplanes.",
    "Satellite view of Hohai University.",
    "A building next to a lake.",
    "Many people in a stadium.",
    "a cute cat",
]
text = tokenizer(text_queries)

In [ ]:
with torch.no_grad():
    image = (
        preprocess(Image.open("/Users/gabriele/Desktop/RemoteCLIP/assets/airport.jpg"))
        .unsqueeze(0)
        .to("mps")
    )
    image_features = model.encode_image(image)
    text_features = model.encode_text(text.to("mps"))
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1).cpu().numpy()[0]

print(f"Predictions of {model_name}:")
for query, prob in zip(text_queries, text_probs):
    print(f"{query:<40} {prob * 100:5.1f}%")

### Check if preprocessing match

In [ ]:
import numpy as np
import torchvision.transforms as T

arr = np.array(Image.open("/Users/gabriele/Desktop/RemoteCLIP/assets/airport.jpg"))
img = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).float()
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)
img = img / 255
img = img.clip(0, 1)

In [ ]:
resize_crop = T.Compose(
    [
        T.Resize(224, interpolation=T.InterpolationMode.BICUBIC, antialias=True),
        T.CenterCrop(224),
    ]
)
normalize = T.Normalize(mean=CLIP_MEAN, std=CLIP_STD)

img = resize_crop(img)
img = normalize(img)

In [ ]:
image = preprocess(Image.open("/Users/gabriele/Desktop/RemoteCLIP/assets/airport.jpg")).unsqueeze(
    0
)

In [ ]:
(image == img).all()